Selecting the 50-example baseline set from MTSamples

**Goal:** pick exactly 50 MTSamples reports for the single-Gemini baseline, diverse across
document type, clinical specialty, and length.

#### 1. Load the data

In [5]:
import re
import hashlib
import pandas as pd
from sklearn.model_selection import train_test_split

# Path to the MTSamples CSV
CSV_PATH = '../data/mtsamples.csv'

# Fixed random seed so sample set stays consistent
SEED = 42

# The CSV's first column is an unnamed row index; use it as a stable report ID
df = pd.read_csv(CSV_PATH, index_col=0).rename_axis("row_id").reset_index()

# View dataset
df.head(5)

,row_id,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


#### 2. Remove empty rows and duplicates

MTSamples lists the same report under several specialties so we must collapse these into one row with the lsited specialities as a separate column

In [6]:
# Collapse repeated whitespace so near-identical copies match exactly
df["text"] = (
    df["transcription"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Specialty labels come with a leading space (" Radiology"), which breaks exact matching
df["medical_specialty"] = df["medical_specialty"].str.strip()
df["sample_name"] = df["sample_name"].str.strip()

# Drop rows with no transcription text at all
df = df[df["text"] != ""]

# One row per unique report
reports = (
    df.groupby("text")
    .agg(
        row_id=("row_id", "first"),           
        sample_name=("sample_name", "first"),
        specialties=("medical_specialty", lambda s: sorted(set(s))),
    )
    .reset_index()
)

print(f"{len(df)} rows -> {len(reports)} unique reports")

4966 rows -> 2357 unique reports


#### 3. Remove report types and documents that don't fit the task

- **Excluded categories:** autopsies, legal/work-comp exams, letters, diet handouts, and pathology reports.
- **Transcription gaps:** There are entries with omitted words due to transcription errors.
- **Templates:** a few documents are blank "normal exam" templates.

In [7]:
EXCLUDED_SPECIALTIES = {
    "Autopsy",
    "IME-QME-Work Comp etc.",
    "Letters",
    "Diets and Nutritions",
    "Lab Medicine - Pathology",
}

# Drop a report if ANY of its specialty labels is in the excluded list
is_excluded = reports["specialties"].apply(lambda labels: len(set(labels) & EXCLUDED_SPECIALTIES) > 0)

# Drop reports with blanks in the text
has_blanks = reports["text"].str.contains("___", regex=False)

# Drop templates (the word "template" appears in the name or text)
is_template = (
    reports["sample_name"].str.contains("template", case=False)
    | reports["text"].str.contains("template", case=False)
)

print("Excluded category:", is_excluded.sum())
print("Has blanks:       ", has_blanks.sum())
print("Template:         ", is_template.sum())

reports = reports[~is_excluded & ~has_blanks & ~is_template]
print("Remaining:", len(reports))

Excluded category: 63
Has blanks:        190
Template:          38
Remaining: 2068


#### 4. Keep reports of a reasonable size

- Range: 150 - 800 words

In [8]:
MIN_WORDS = 150
MAX_WORDS = 800

reports["n_words"] = reports["text"].str.split().str.len()
reports = reports[reports["n_words"].between(MIN_WORDS, MAX_WORDS)]

print("Remaining:", len(reports))
reports["n_words"].describe()

Remaining: 1629


count    1629.000000
mean      401.888889
std       171.536640
min       150.000000
25%       256.000000
50%       378.000000
75%       527.000000
max       798.000000
Name: n_words, dtype: float64

#### 5. Label each report's document type and clinical domain

The `medical_specialty` column mixes two different things:
- **Document type** — Discharge Summary, Radiology, SOAP, Consult, ER…
- **Clinical domain** — Cardiovascular, Orthopedic, Neurology…

We split them into two columns, because we want diversity on both.
For document type we check labels in a fixed priority order, so each report gets exactly one.

In [9]:
def get_doc_type(labels, text):
    labels = set(labels)
    # Checked in priority order: the first match wins
    if "Discharge Summary" in labels or "DISCHARGE DIAGNOS" in text:
        return "Discharge summary"
    if "Emergency Room Reports" in labels:
        return "ER report"
    if "Radiology" in labels:
        return "Radiology report"
    if "SOAP / Chart / Progress Notes" in labels or "Office Notes" in labels:
        return "SOAP / progress / office"
    if "Consult - History and Phy." in labels:
        return "Consult / H&P"
    if "Surgery" in labels or "PROCEDURE" in text.upper():
        return "Op / procedure note"
    return "Other"


# Labels that describe the document type (or are too generic), so they don't count as a domain
NOT_A_DOMAIN = {
    "Discharge Summary", "Emergency Room Reports", "Radiology",
    "SOAP / Chart / Progress Notes", "Office Notes",
    "Consult - History and Phy.", "Surgery", "General Medicine",
}


def get_domain(labels):
    # The first label that names an actual clinical domain; otherwise "General Medicine"
    for label in labels:
        if label not in NOT_A_DOMAIN:
            return label
    return "General Medicine"


reports["doc_type"] = [get_doc_type(l, t) for l, t in zip(reports["specialties"], reports["text"])]
reports["domain"] = reports["specialties"].apply(get_domain)

# Drop the few reports that didn't fit any document type
reports = reports[reports["doc_type"] != "Other"]

reports["doc_type"].value_counts()

doc_type
Op / procedure note         753
Consult / H&P               337
Radiology report            192
SOAP / progress / office    151
Discharge summary            90
ER report                    56
Name: count, dtype: int64

#### 6. Flag content types we care about (keyword checks)

The November error analysis asks which content types cause hallucinations (drug names/doses,
lab values, procedures). We flag them now so we can make sure the 50 contain enough of each.
These are simple keyword checks, not perfect — they're just for checking coverage.

In [10]:
# A number followed by a dose unit, e.g. "40 mg", "0.5 mL", "10 units"
MED_PATTERN = r"\b\d+(?:\.\d+)?\s?(?:mg|mcg|mL|units?)\b"

# Common lab tests and vital signs
LAB_PATTERN = (
    r"\b(?:blood pressure|BP|heart rate|pulse|temperature|hemoglobin|WBC|white count|"
    r"platelets?|creatinine|BUN|glucose|sodium|potassium|INR|troponin)\b"
)

reports["has_meds"] = reports["text"].str.contains(MED_PATTERN, case=False, regex=True)
reports["has_labs"] = reports["text"].str.contains(LAB_PATTERN, case=False, regex=True)

# Short / medium / long, measured WITHIN each document type (discharge summaries run longer than radiology reports)
reports["length_bin"] = reports.groupby("doc_type")["n_words"].transform(
    lambda x: pd.qcut(x, 3, labels=False)
).map({0: "short", 1: "medium", 2: "long"})

reports[["has_meds", "has_labs"]].mean().round(2)

has_meds    0.43
has_labs    0.37
dtype: float64

#### 7. Pick 50 using fixed quotas

1. Shuffle the pool with the fixed seed.
2. Within each document type, alternate short → medium → long, so lengths are spread out.
3. Walk down the list and take reports until the quota is filled, skipping any report whose clinical domain already has `MAX_PER_DOMAIN` picks.

In [11]:
QUOTAS = {
    "Discharge summary": 10,
    "Consult / H&P": 9,
    "Op / procedure note": 9,
    "Radiology report": 8,
    "SOAP / progress / office": 8,
    "ER report": 6,
}
assert sum(QUOTAS.values()) == 50

# No single clinical domain (e.g. Cardiology) can take up more than this many of the 50
MAX_PER_DOMAIN = 6

# Reports rejected during manual review: {row_id: "reason"}. Starts empty; fill it in during Step 9.
REJECTED = {
    # 1234: "garbled text",
}

# Remove rejected reports, then shuffle with the fixed seed
pool = reports[~reports["row_id"].isin(REJECTED.keys())]
pool = pool.sample(frac=1, random_state=SEED)

# Give each report its position within its (doc type, length bin) group: 0, 1, 2, ...
# Sorting by that position interleaves short/medium/long. kind="stable" keeps the shuffled order for ties.
pool["position"] = pool.groupby(["doc_type", "length_bin"]).cumcount()
pool = pool.sort_values("position", kind="stable")

selected_ids = []
domain_counts = {}

for doc_type, quota in QUOTAS.items():
    candidates = pool[pool["doc_type"] == doc_type]
    n_picked = 0

    for _, row in candidates.iterrows():
        if n_picked == quota:
            break
        # Skip if this domain is already full
        if domain_counts.get(row["domain"], 0) >= MAX_PER_DOMAIN:
            continue
        selected_ids.append(row["row_id"])
        domain_counts[row["domain"]] = domain_counts.get(row["domain"], 0) + 1
        n_picked += 1

    if n_picked < quota:
        print(f"WARNING: only found {n_picked}/{quota} for {doc_type}")

selected = pool[pool["row_id"].isin(selected_ids)].copy()
print("Selected:", len(selected))

Selected: 50


#### 8. Check diversity

In [12]:
print("Document types:")
print(selected["doc_type"].value_counts(), "\n")

print(f"Clinical domains ({selected['domain'].nunique()} distinct, target >= 12):")
print(selected["domain"].value_counts(), "\n")

print("Length bins:")
print(selected["length_bin"].value_counts(), "\n")

print("Has medication + dose:", selected["has_meds"].sum(), "(target >= 20)")
print("Has labs / vitals:    ", selected["has_labs"].sum(), "(target >= 15)")

Document types:
doc_type
Discharge summary           10
Op / procedure note          9
Consult / H&P                9
Radiology report             8
SOAP / progress / office     8
ER report                    6
Name: count, dtype: int64 

Clinical domains (16 distinct, target >= 12):
domain
Gastroenterology              6
Cardiovascular / Pulmonary    6
Neurology                     6
General Medicine              6
Urology                       6
Obstetrics / Gynecology       4
ENT - Otolaryngology          3
Orthopedic                    3
Hematology - Oncology         2
Podiatry                      2
Pediatrics - Neonatal         1
Speech - Language             1
Bariatrics                    1
Nephrology                    1
Endocrinology                 1
Dermatology                   1
Name: count, dtype: int64 

Length bins:
length_bin
medium    18
short     17
long      15
Name: count, dtype: int64 

Has medication + dose: 24 (target >= 20)
Has labs / vitals:     27 (target >=

#### 9. Manually review the 50

Read each selected report (the cell below prints them one at a time). Reject a report if it is:
- garbled or cut off,
- essentially a template or boilerplate,
- something a patient would never need explained.

For each rejection, add its `row_id` and a reason to `REJECTED` in Step 7, then rerun from
Step 7 down. The replacement comes from the same document type automatically, and the reasons
stay in the notebook as a record of the curation.

In [13]:
# Change i from 0 to 49 to read through the selected reports
i = 0

row = selected.iloc[i]
print(f"[{i}] row_id={row['row_id']} | {row['doc_type']} | {row['domain']} | {row['n_words']} words")
print(row["sample_name"], "\n")
print(row["text"])

[0] row_id=858 | Op / procedure note | Gastroenterology | 210 words
Endoscopy - 4 

PROCEDURE: , Endoscopy.,CLINICAL INDICATIONS: , Intermittent rectal bleeding with abdominal pain.,ANESTHESIA: , Fentanyl 100 mcg and 5 mg of IV Versed.,PROCEDURE:, The patient was taken to the GI lab and placed in the left lateral supine position. Continuous pulse oximetry and blood pressure monitoring were in place. After informed consent was obtained, the video endoscope was inserted over the dorsum of the tongue without difficulty. With swallowing, the scope was advanced down the esophagus into the body of the stomach. The scope was further advanced down to the antrum and through the pylorus into the duodenum, which was visualized into its second portion. It appeared free of stricture, neoplasm, or ulceration. Samples were obtained from the antrum and prepyloric area to check for Helicobacter, rapid urease, and additional samples were sent to pathology. Retroflexion view of the fundus of the stomach 

#### 10. Split into dev / test and freeze

- **Dev (10):** use these while writing and tuning prompts.
- **Test (40):** don't look at outputs on these until a prompt is final. If we tune prompts on
  all 50, both the baseline and the multi-agent scores will be inflated, and the comparison
  between them means less.

The split is stratified, so dev and test have similar document-type mixes.

**Two output files:**
- `baseline_50_ids.csv` — IDs, a text hash, and labels, but **no report text**. Commit this one.
  (The project brief says not to re-post raw MTSamples records in the public repo.)
- `baseline_50_full.csv`

The `text_hash` lets anyone confirm they've rebuilt exactly the same text from `mtsamples.csv`.

In [14]:
dev, test = train_test_split(
    selected,
    test_size=40,
    stratify=selected["doc_type"],
    random_state=SEED,
)
selected["split"] = "test"
selected.loc[dev.index, "split"] = "dev"

# Short fingerprint of each report's text, to verify the frozen set hasn't silently changed
selected["text_hash"] = selected["text"].apply(lambda t: hashlib.md5(t.encode()).hexdigest()[:12])

selected["specialties"] = selected["specialties"].apply(" | ".join)

columns = ["row_id", "text_hash", "split", "doc_type", "domain", "specialties",
           "n_words", "length_bin", "has_meds", "has_labs", "sample_name"]

selected = selected.sort_values(["split", "doc_type", "row_id"])
selected[columns].to_csv("../data/baseline_50_ids.csv", index=False)
selected[columns + ["text"]].to_csv("../data/baseline_50_full.csv", index=False)

print(selected["split"].value_counts())
selected[columns].head(10)

split
test    40
dev     10
Name: count, dtype: int64


,row_id,text_hash,split,doc_type,domain,specialties,n_words,length_bin,has_meds,has_labs,sample_name
745,1896,009354afe6a4,dev,Consult / H&P,Gastroenterology,Consult - History and Phy. | Gastroenterology ...,410,medium,False,True,Rectal Bleeding - 1-year-old
1790,3527,11d9788ea826,dev,Consult / H&P,Gastroenterology,Consult - History and Phy. | Gastroenterology,289,short,False,True,GI Consultation - 1
49,1910,6946760e5a29,dev,Discharge summary,ENT - Otolaryngology,Discharge Summary | ENT - Otolaryngology | Ped...,633,long,True,True,Otitis Media - Discharge Summary
46,1916,5b0597954a7f,dev,Discharge summary,Pediatrics - Neonatal,Discharge Summary | Pediatrics - Neonatal,600,long,True,True,Neonatal Discharge Summary - 1
227,39,b9d5273bb048,dev,ER report,Urology,Emergency Room Reports | Urology,622,long,False,False,Testicular Pain
1180,540,3590dd598e52,dev,Op / procedure note,Orthopedic,Orthopedic | Surgery,641,long,False,True,Mumford Procedure & Acromioplasty
1233,620,c677eef26160,dev,Op / procedure note,Obstetrics / Gynecology,Obstetrics / Gynecology | Surgery,442,medium,False,False,Laparoscopic Supracervical Hysterectomy.
545,1656,760dd491ed0c,dev,Radiology report,Cardiovascular / Pulmonary,Cardiovascular / Pulmonary | Radiology,174,short,False,False,Echocardiogram - 2
99,1730,b210f4471cfe,dev,Radiology report,Neurology,Neurology | Radiology,450,long,False,True,Cerebral Angiogram - Lateral Medullary Syndrome
344,1355,b1533f0c1df9,dev,SOAP / progress / office,Hematology - Oncology,Hematology - Oncology | SOAP / Chart / Progres...,164,short,True,True,Lung Cancer Followup
